# Palindrome Sentences with HuggingFace Transformers

This notebook explores three strategies for generating palindrome sentences using the HuggingFace `transformers` library:

1. **Few-shot prompting** — ask an instruction-tuned model and see what happens (baseline)
2. **Half-and-mirror with `LogitsProcessor`** — generate the first half freely, then constrain the second half to be the character mirror
3. **BERT bidirectional quality scoring** — use a masked language model to score how natural both directions of a palindrome sound

A palindrome reads the same forwards and backwards (ignoring spaces and punctuation):
```
never odd or even
▲                ▲
same characters reversed
```


In [ ]:
# Install required packages (skip if already installed)
# !pip install transformers torch accelerate sentencepiece

import re
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BertTokenizer,
    BertForMaskedLM,
    LogitsProcessor,
    pipeline,
)

print(f"torch version : {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device  : {device}")


## 1. Palindrome Utilities

In [ ]:
def clean_text(text: str) -> str:
    """Remove all non-alphabetic characters and lowercase."""
    return re.sub(r'[^a-z]', '', text.lower())

def is_palindrome(text: str) -> bool:
    c = clean_text(text)
    return c == c[::-1]

def palindrome_distance(text: str) -> int:
    """Count character positions that differ from their mirror. 0 = perfect."""
    c = clean_text(text)
    return sum(1 for i in range(len(c) // 2) if c[i] != c[-(i+1)])

# Sanity checks
for example in [
    "A man a plan a canal Panama",
    "Never odd or even",
    "Was it a car or a cat I saw",
    "This is not a palindrome",
]:
    ok = is_palindrome(example)
    dist = palindrome_distance(example)
    print(f"  {'OK' if ok else 'XX'}  [{dist:2d} errors]  {example}")


## 2. The Tokenisation Asymmetry Problem

BPE tokenisers process text **greedily left-to-right**.
The token sequence for a string and its character-reversed version are usually completely different.
This is a fundamental reason why palindrome generation is non-trivial for LLMs.


In [ ]:
gpt2_tok = AutoTokenizer.from_pretrained("gpt2")

examples = [
    "never odd or even",
    "A man a plan a canal Panama",
    "racecar",
]

print("=" * 65)
for text in examples:
    rev = text[::-1]
    fwd_tokens = gpt2_tok.tokenize(text)
    bwd_tokens = gpt2_tok.tokenize(rev)
    is_token_palindrome = (fwd_tokens == list(reversed(bwd_tokens)))
    print(f"\nOriginal : {text}")
    print(f"  Forward tokens : {fwd_tokens}")
    print(f"  Reversed text  : {rev}")
    print(f"  Reversed tokens: {bwd_tokens}")
    print(f"  Token palindrome? {is_token_palindrome}")

print("\nConclusion: reversing a character palindrome does NOT reverse its token sequence.")
print("We must work at the character level and accept the token asymmetry.")


## 3. Approach 1: Few-Shot Prompting (Baseline)

Load a small causal LM and prompt it with classic palindrome examples.
GPT-2 (117M) is used here for speed; swap in a larger instruct model for better results.


In [ ]:
# GPT-2 is fast but not instruction-tuned.
# For better results try:
#   "mistralai/Mistral-7B-Instruct-v0.3"
#   "meta-llama/Llama-3.2-3B-Instruct"
model_name = "gpt2"

print(f"Loading {model_name}...")
generator = pipeline(
    "text-generation",
    model=model_name,
    device=0 if torch.cuda.is_available() else -1,
)
print("Ready.")


In [ ]:
few_shot_prompt = (
    "A palindrome is a sentence that reads the same forwards and backwards.\n\n"
    "Examples:\n"
    "- A man a plan a canal Panama\n"
    "- Never odd or even\n"
    "- Was it a car or a cat I saw\n"
    "- Step on no pets\n\n"
    "New palindrome:\n- "
)

results = generator(
    few_shot_prompt,
    max_new_tokens=30,
    num_return_sequences=5,
    do_sample=True,
    temperature=0.9,
    pad_token_id=generator.tokenizer.eos_token_id,
)

print("Generated candidates:")
for i, r in enumerate(results, 1):
    text = r["generated_text"][len(few_shot_prompt):].strip().split("\n")[0]
    ok = is_palindrome(text)
    dist = palindrome_distance(text)
    label = "PALINDROME!" if ok else f"{dist} errors"
    print(f"  {i}. [{label:15s}] {text}")

print("\nObservation: GPT-2 rarely generates correct character-level palindromes.")
print("The global constraint is beyond greedy left-to-right decoding without guidance.")


## 4. Approach 2: Half-and-Mirror with a Custom LogitsProcessor

Instead of hoping the model satisfies the palindrome constraint, we **enforce it**.

**Strategy:**
1. Tokenise a seed phrase as the prompt (the "first half")
2. After the prompt, use a `LogitsProcessor` to allow only tokens whose leading characters match the required mirror
3. This guarantees a character palindrome — the second half is deterministic

The creative challenge is choosing good first-half seeds.


In [ ]:
class MirrorSuffixLogitsProcessor(LogitsProcessor):
    """
    Forces the LM to emit a character-mirror of a given character sequence.

    Once activated (after `offset` new tokens), each step allows only tokens
    whose first decoded character matches the next required mirror character.

    Args:
        tokenizer   : the model tokenizer
        mirror_chars: character sequence that must be emitted (reversed prefix letters)
        offset      : how many newly generated tokens to pass through freely before enforcing
    """
    def __init__(self, tokenizer, mirror_chars: str, offset: int = 0):
        self.tokenizer = tokenizer
        self.mirror_chars = mirror_chars.lower()
        self.offset = offset
        self._call_count = 0
        self._char_cursor = 0

        # Pre-index vocab: first_char -> list of token ids
        self._char_to_ids: dict = {}
        for tok_str, tok_id in tokenizer.get_vocab().items():
            # Strip BPE / SentencePiece space markers (Ġ = GPT-2, ▁ = SentencePiece)
            clean = tok_str.lstrip("\u0120\u2581 ").lower()
            if clean:
                self._char_to_ids.setdefault(clean[0], []).append(tok_id)

    def __call__(self, input_ids, scores):
        self._call_count += 1
        if self._call_count <= self.offset:
            return scores  # free generation

        if self._char_cursor >= len(self.mirror_chars):
            return scores  # mirror exhausted

        target = self.mirror_chars[self._char_cursor]
        self._char_cursor += 1

        allowed = self._char_to_ids.get(target, [])
        if not allowed:
            return scores  # no vocab match, let model decide

        new_scores = torch.full_like(scores, float("-inf"))
        for tok_id in allowed:
            new_scores[:, tok_id] = scores[:, tok_id]
        return new_scores

print("MirrorSuffixLogitsProcessor ready.")


In [ ]:
# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
lm = AutoModelForCausalLM.from_pretrained("gpt2").to(device)
lm.eval()

def mirror_generate(seed: str, lm, tokenizer, device) -> str:
    """Generate a character palindrome by mirroring `seed`."""
    seed_chars = clean_text(seed)
    mirror = seed_chars[::-1]

    input_ids = tokenizer.encode(seed, return_tensors="pt").to(device)
    prompt_len = input_ids.shape[1]

    processor = MirrorSuffixLogitsProcessor(
        tokenizer=tokenizer,
        mirror_chars=mirror,
        offset=0,
    )

    with torch.no_grad():
        output = lm.generate(
            input_ids,
            max_new_tokens=len(mirror) + 3,
            logits_processor=[processor],
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    suffix = tokenizer.decode(output[0][prompt_len:], skip_special_tokens=True)
    return seed + suffix

# Test with several seeds
seeds = ["step on", "never odd", "was it a car", "go dog", "do geese"]
print(f"{'Seed':<18} {'Result':<38} Status")
print("-" * 72)
for seed in seeds:
    result = mirror_generate(seed, lm, tokenizer, device)
    ok = is_palindrome(result)
    dist = palindrome_distance(result)
    status = "PALINDROME" if ok else f"dist={dist}"
    print(f"{seed:<18} {result:<38} {status}")


## 5. Approach 3: BERT Bidirectional Quality Scoring

Mirror-generation guarantees correctness but not naturalness.
To find palindromes that sound good **in both directions**, we use BERT's masked language modelling (MLM).

BERT reads the entire context bidirectionally when predicting a masked token, making its
pseudo-log-likelihood (PLL) a strong naturalness signal.
We score each palindrome candidate in both directions and keep the best.


In [ ]:
print("Loading bert-base-uncased...")
bert_tok = BertTokenizer.from_pretrained("bert-base-uncased")
bert_model = BertForMaskedLM.from_pretrained("bert-base-uncased").to(device)
bert_model.eval()
print("BERT ready.")


In [ ]:
def bert_pll(text: str) -> float:
    """
    Pseudo-log-likelihood: mask each token, sum log P(token | all others).
    Returns mean log probability. Higher = more natural.
    """
    inputs = bert_tok(text, return_tensors="pt").to(device)
    ids = inputs["input_ids"]
    total, count = 0.0, 0
    for i in range(1, ids.size(1) - 1):  # skip [CLS] and [SEP]
        masked = ids.clone()
        masked[0, i] = bert_tok.mask_token_id
        with torch.no_grad():
            logits = bert_model(masked).logits
        lp = torch.log_softmax(logits[0, i], dim=-1)
        total += lp[ids[0, i]].item()
        count += 1
    return total / count if count > 0 else float("-inf")

def score_both_directions(text: str) -> dict:
    """Score the palindrome forward and as character-reversed."""
    chars = clean_text(text)
    rev_spaced = " ".join(chars[i : i + 4] for i in range(0, len(chars), 4))
    fwd = bert_pll(text)
    bwd = bert_pll(rev_spaced)
    return {
        "text": text,
        "palindrome": is_palindrome(text),
        "fwd": fwd,
        "bwd": bwd,
        "joint": (fwd + bwd) / 2,
    }

# Score known good palindromes vs a non-palindrome baseline
test_texts = [
    "A man a plan a canal Panama",
    "Never odd or even",
    "Was it a car or a cat I saw",
    "Step on no pets",
    "Do geese see God",
    "Today is a nice sunny day",   # non-palindrome baseline
]
print(f"{'Text':<35} {'Pal':5} {'Fwd':>8} {'Bwd':>8} {'Joint':>8}")
print("-" * 70)
for t in test_texts:
    s = score_both_directions(t)
    print(f"{t:<35} {str(s['palindrome']):5} {s['fwd']:8.3f} {s['bwd']:8.3f} {s['joint']:8.3f}")


In [ ]:
# Full pipeline: generate many candidates, score with BERT, rank by joint score

seeds_batch = [
    "step on", "never", "was it", "do geese", "go dog",
    "live evil", "race a car", "star rats", "name no one",
    "level", "civic", "noon", "rotator", "deed",
]

print("Running full pipeline (generate + score)...")
print()

results = []
for seed in seeds_batch:
    try:
        candidate = mirror_generate(seed, lm, tokenizer, device)
        if is_palindrome(candidate):
            s = score_both_directions(candidate)
            results.append(s)
    except Exception as e:
        print(f"  Error on '{seed}': {e}")

results.sort(key=lambda x: x["joint"], reverse=True)

print(f"{'Rank':<5} {'Palindrome':<38} {'Fwd':>8} {'Bwd':>8} {'Joint':>8}")
print("-" * 72)
for rank, r in enumerate(results, 1):
    print(f"{rank:<5} {r['text']:<38} {r['fwd']:8.3f} {r['bwd']:8.3f} {r['joint']:8.3f}")


## 6. Summary

| Approach | Guarantees palindrome? | Naturally readable? | Complexity |
|---|---|---|---|
| Few-shot prompting | No | Sometimes | Low |
| Half-and-mirror (LogitsProcessor) | Yes | Second half mechanical | Medium |
| Half-and-mirror + BERT ranking | Yes (filtered) | Better (filtered) | Higher |

**Key takeaways:**
- Standard autoregressive generation cannot satisfy the palindrome constraint without external guidance
- `LogitsProcessor` is the correct hook to encode character-level constraints at decode time
- BERT's bidirectionality makes it a natural quality judge for both directions of a palindrome
- The most readable results come from short, well-chosen seeds — the creative task is seed selection

**Next steps:** See `palindrome_ollama.ipynb` for the Ollama-based iterative refinement approach.
